In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
%pip install -q diffusers transformers accelerate \
    huggingface_hub safetensors sentencepiece \
    imageio imageio-ffmpeg ftfy psutil

In [ ]:
import os
import gc
import json
import time
import hashlib
import shutil
import psutil
import torch

from pathlib import Path
from datetime import datetime, timezone
from PIL import Image, ImageOps
from IPython.display import display, Video

# Model configuration
MODEL_ID = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"

CACHE_DIR = Path("/tmp/heritage-models/wan")

# Dataset and output paths
INPUT_DIR = Path(
    "/kaggle/input/datasets/bahailuabera/wan-inputs"
)

PROJECT_DIR = Path(
    "/kaggle/working/heritage-short"
)

KEYFRAMES_DIR = INPUT_DIR / "outputs/keyframes"
SHOTS_DIR = PROJECT_DIR / "outputs/shots"

LOG_PATH = PROJECT_DIR / "wan_runs.jsonl"

CACHE_DIR.mkdir(parents=True, exist_ok=True)
SHOTS_DIR.mkdir(parents=True, exist_ok=True)

# Load storyboard
storyboard = json.loads(
    (INPUT_DIR / "storyboard.json").read_text(
        encoding="utf-8"
    )
)

for shot in storyboard["shots"]:
    path = KEYFRAMES_DIR / f"{shot['id']}.png"

    assert path.is_file(), f"Missing keyframe: {path}"

# Verify hardware
assert torch.cuda.is_available(), "CUDA unavailable"

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    gpu = torch.cuda.get_device_properties(i)

    print(
        f"GPU {i}: {gpu.name} | "
        f"{gpu.total_memory / 1024**3:.2f} GiB"
    )

print(
    "Available RAM:",
    round(psutil.virtual_memory().available / 1024**3, 2),
    "GiB"
)

print(
    "Model cache disk space:",
    round(shutil.disk_usage(CACHE_DIR).free / 1024**3, 2),
    "GiB"
)

print("\nFilm:", storyboard["film"]["title"])
print("Shots:", len(storyboard["shots"]))
print("Keyframes:", len(list(KEYFRAMES_DIR.glob("*.png"))))

In [ ]:
from diffusers import (
    AutoencoderKLWan,
    WanImageToVideoPipeline,
)

from diffusers.utils import export_to_video

gc.collect()
torch.cuda.empty_cache()

start = time.perf_counter()

print("Loading VAE...", flush=True)

vae = AutoencoderKLWan.from_pretrained(
    MODEL_ID,
    subfolder="vae",
    torch_dtype=torch.float32,
    cache_dir=str(CACHE_DIR),
    low_cpu_mem_usage=True,
)

print("Loading Wan pipeline...", flush=True)

pipe = WanImageToVideoPipeline.from_pretrained(
    MODEL_ID,
    vae=vae,
    torch_dtype=torch.float16,
    cache_dir=str(CACHE_DIR),
    low_cpu_mem_usage=True,
    image_encoder=None,
    image_processor=None,
    expand_timesteps=True,
)

print("Enabling sequential CPU offloading...", flush=True)

pipe.enable_sequential_cpu_offload(gpu_id=0)

gc.collect()

print("\nModel loaded successfully")

print(
    "Loading time:",
    round(time.perf_counter() - start, 2),
    "seconds",
)

print(
    "Available RAM:",
    round(psutil.virtual_memory().available / 1024**3, 2),
    "GiB",
)

In [ ]:
# ==========================================
# GENERATION CONFIGURATION
# ==========================================
SELECTED_SHOTS = [
    "shot_002",
    "shot_006",
    "shot_009",
    "shot_010",
]

WIDTH = 512
HEIGHT = 288
NUM_FRAMES = 49
FPS = 16
STEPS = 10
GUIDANCE_SCALE = 1.0

FORCE = False

# ==========================================
# GENERATION IMPLEMENTATION
# ==========================================

def hash_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def make_cache_key(parameters):
    return hashlib.sha256(
        json.dumps(
            parameters,
            sort_keys=True,
            ensure_ascii=False,
        ).encode("utf-8")
    ).hexdigest()


def write_log(record):
    with LOG_PATH.open("a", encoding="utf-8") as file:
        file.write(
            json.dumps(record, ensure_ascii=False) + "\n"
        )


def generate_shot(shot):

    shot_id = shot["id"]

    keyframe_path = KEYFRAMES_DIR / f"{shot_id}.png"
    output = SHOTS_DIR / f"{shot_id}.mp4"
    temporary_output = SHOTS_DIR / f"{shot_id}.tmp.mp4"
    manifest_path = SHOTS_DIR / f"{shot_id}.json"

    prompt = (
        f"{shot['image_prompt']}. "
        f"{shot['motion_prompt']}. "
        "Preserve the original characters' ages, "
        "facial features, body proportions and clothing. "
        "Natural cinematic movement, consistent lighting, "
        "stable environment and realistic human motion."
    )

    negative_prompt = (
        "Changing character identity, changing character age, "
        "distorted faces, extra limbs, deformed hands, "
        "flickering, unnatural movement, warped background, "
        "blurry image, text, subtitles"
    )

    parameters = {
        "model": MODEL_ID,
        "width": WIDTH,
        "height": HEIGHT,
        "num_frames": NUM_FRAMES,
        "fps": FPS,
        "num_inference_steps": STEPS,
        "guidance_scale": GUIDANCE_SCALE,
        "precision": "float16",
        "vae_precision": "float32",
        "offload_strategy": "sequential",
        "prompt": prompt,
        "negative_prompt": (
            negative_prompt
            if GUIDANCE_SCALE > 1.0
            else None
        ),
        "seed": shot["seed"],
        "keyframe_sha256": hash_file(keyframe_path),
    }

    cache_key = make_cache_key(parameters)

    # Check cached artifact.
    if (
        not FORCE
        and output.is_file()
        and output.stat().st_size > 0
        and manifest_path.is_file()
    ):
        try:
            manifest = json.loads(
                manifest_path.read_text()
            )

            if (
                manifest["cache_key"] == cache_key
                and manifest["output_sha256"] == hash_file(output)
            ):
                print(f"[CACHE HIT] {shot_id}")

                write_log({
                    "timestamp": datetime.now(
                        timezone.utc
                    ).isoformat(),
                    "event": "cache_hit",
                    "stage": "shots",
                    "shot_id": shot_id,
                    "cost_usd": 0,
                })

                return output

        except (ValueError, KeyError, OSError):
            pass

    print(f"\n[GENERATING] {shot_id}", flush=True)

    image = Image.open(keyframe_path).convert("RGB")

    image = ImageOps.fit(
        image,
        (WIDTH, HEIGHT),
        method=Image.Resampling.LANCZOS,
    )

    display(image)

    generator = torch.Generator(device="cpu")

    if shot["seed"] is not None:
        generator.manual_seed(shot["seed"])
    else:
        generator.seed()

    parameters["effective_seed"] = generator.initial_seed()

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(0)

    start = time.perf_counter()

    def on_step_end(
        pipeline,
        step_index,
        timestep,
        callback_kwargs,
    ):
        elapsed = time.perf_counter() - start

        allocated = (
            torch.cuda.memory_allocated(0) / 1024**3
        )

        print(
            f"{shot_id} | "
            f"Step {step_index + 1}/{STEPS} | "
            f"Elapsed: {elapsed:.1f}s | "
            f"VRAM: {allocated:.2f} GiB",
            flush=True,
        )

        return callback_kwargs

    record = {
        "timestamp": datetime.now(
            timezone.utc
        ).isoformat(),
        "event": "model_call",
        "stage": "shots",
        "provider": "kaggle",
        "model": MODEL_ID,
        "shot_id": shot_id,
        "seed": parameters["effective_seed"],
        "parameters": parameters,
        "cache_key": cache_key,
        "gpu": torch.cuda.get_device_name(0),
        "gpu_index": 0,
        "cost_usd": None,
        "estimated_cost_usd": None,
        "status": "running",
    }

    try:
        print("Starting inference...", flush=True)

        with torch.inference_mode():
            frames = pipe(
                image=image,
                prompt=prompt,
                negative_prompt=parameters["negative_prompt"],
                width=WIDTH,
                height=HEIGHT,
                num_frames=NUM_FRAMES,
                num_inference_steps=STEPS,
                guidance_scale=GUIDANCE_SCALE,
                generator=generator,
                callback_on_step_end=on_step_end,
            ).frames[0]

        torch.cuda.synchronize(0)

        print("Exporting video...", flush=True)

        export_to_video(
            frames,
            str(temporary_output),
            fps=FPS,
        )

        if temporary_output.stat().st_size == 0:
            raise RuntimeError("Generated video is empty")

        temporary_output.replace(output)

        manifest = {
            "shot_id": shot_id,
            "cache_key": cache_key,
            "output_sha256": hash_file(output),
            "parameters": parameters,
            "duration_seconds": len(frames) / FPS,
            "backend": "wan",
        }

        temporary_manifest = manifest_path.with_suffix(".tmp")

        temporary_manifest.write_text(
            json.dumps(manifest, indent=2),
            encoding="utf-8",
        )

        temporary_manifest.replace(manifest_path)

        record["status"] = "success"
        record["output_path"] = str(output)
        record["duration_seconds"] = len(frames) / FPS

        del frames

    except BaseException as error:
        record["status"] = "failed"
        record["error_type"] = type(error).__name__
        record["error"] = str(error)

        raise

    finally:
        elapsed = time.perf_counter() - start

        record["latency_seconds"] = round(elapsed, 3)

        record["gpu_minutes"] = round(
            elapsed / 60, 3
        )

        record["peak_vram_gib"] = round(
            torch.cuda.max_memory_allocated(0) / 1024**3,
            3,
        )

        write_log(record)

        temporary_output.unlink(missing_ok=True)

        print(json.dumps(record, indent=2))

        gc.collect()
        torch.cuda.empty_cache()

    return output


# ==========================================
# EXECUTE SELECTED SHOTS
# ==========================================

available_shots = {
    shot["id"]: shot
    for shot in storyboard["shots"]
}

unknown = set(SELECTED_SHOTS) - set(available_shots)

assert not unknown, f"Unknown shots: {unknown}"

generated = []

for shot_id in SELECTED_SHOTS:
    output = generate_shot(
        available_shots[shot_id]
    )

    generated.append(output)

print("\nGeneration completed")

for output in generated:
    print(output)

In [ ]:
import zipfile

videos = sorted(SHOTS_DIR.glob("shot_*.mp4"))

print("Generated videos:", len(videos))

for path in videos:
    print(path.name)
    display(
        Video(
            str(path),
            embed=True,
            width=640,
        )
    )

bundle_path = Path(
    "/kaggle/working/wan_outputs.zip"
)

with zipfile.ZipFile(
    bundle_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:

    for path in sorted(SHOTS_DIR.iterdir()):
        if path.suffix in {".mp4", ".json"}:
            archive.write(
                path,
                f"outputs/shots/{path.name}",
            )

    if LOG_PATH.is_file():
        archive.write(
            LOG_PATH,
            "wan_runs.jsonl",
        )

print("\nExport bundle:", bundle_path)

print(
    "Bundle size:",
    round(bundle_path.stat().st_size / 1024**2, 2),
    "MiB",
)

In [ ]:
experiment_dir = (
    PROJECT_DIR / "outputs/experiments/wan"
)

experiment_dir.mkdir(
    parents=True, exist_ok=True
)

shutil.copy2(
    SHOTS_DIR / "shot_002.mp4",
    experiment_dir / "shot_002_low_resolution.mp4",
)

shutil.copy2(
    SHOTS_DIR / "shot_002.json",
    experiment_dir / "shot_002_low_resolution.json",
)

print("Baseline experiment preserved")

In [ ]:
!pip install -q uv

!git clone https://github.com/ace-step/ACE-Step-1.5.git /tmp/ACE-Step-1.5

%cd /tmp/ACE-Step-1.5

!uv python install 3.12

!uv sync --python 3.12

!git rev-parse HEAD

In [ ]:
%%writefile /tmp/ACE-Step-1.5/generate_heritage.py

import json
import time
import shutil
import subprocess
import hashlib
import torch

from pathlib import Path
from datetime import datetime, timezone

from acestep.handler import AceStepHandler

from acestep.inference import (
    GenerationParams,
    GenerationConfig,
    generate_music,
)

ROOT = Path("/tmp/ACE-Step-1.5")

OUTPUT_DIR = Path(
    "/kaggle/working/heritage-music"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

LOG_PATH = OUTPUT_DIR / "music_runs.jsonl"

REVISION = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=ROOT,
    text=True,
).strip()

MODEL = "acestep-v15-turbo"

DURATION = 90

CANDIDATES = [
    {
        "id": "candidate_001",
        "seed": 201,
        "bpm": 75,
        "caption": (
            "Cinematic Ethiopian heritage instrumental "
            "soundtrack. Gentle krar-inspired plucked "
            "strings, soft acoustic piano, warm ambient "
            "textures, subtle traditional percussion. "
            "Nostalgic rural childhood opening, gradually "
            "building emotional determination, hopeful "
            "inspirational ending. Restrained arrangement "
            "for documentary narration. No vocals."
        ),
    },
    {
        "id": "candidate_002",
        "seed": 202,
        "bpm": 72,
        "caption": (
            "Intimate emotional Ethiopian documentary "
            "instrumental. Soft masenqo-inspired bowed "
            "strings, delicate acoustic plucking, "
            "warm piano and atmospheric pads. "
            "Reflective opening, emotional middle, "
            "gentle uplifting resolution. Sparse "
            "arrangement, suitable beneath spoken "
            "narration. No singing or vocals."
        ),
    },
    {
        "id": "candidate_003",
        "seed": 203,
        "bpm": 80,
        "caption": (
            "Inspirational cinematic instrumental score "
            "about an Ethiopian farming childhood and "
            "technological innovation. Warm traditional "
            "acoustic textures evolving into subtle "
            "modern electronic ambience, gentle strings "
            "and restrained percussion. Nostalgic "
            "beginning, hopeful progressive build, "
            "peaceful ending. No vocals."
        ),
    },
]

print("Loading ACE-Step turbo...", flush=True)

handler = AceStepHandler()

initialization = handler.initialize_service(
    project_root=str(ROOT),
    config_path=MODEL,
    device="cuda",
    offload_to_cpu=True,
)

print(initialization, flush=True)

for candidate in CANDIDATES:

    candidate_id = candidate["id"]

    output = OUTPUT_DIR / f"{candidate_id}.wav"

    manifest = OUTPUT_DIR / f"{candidate_id}.json"

    parameters = {
        **candidate,
        "model": MODEL,
        "revision": REVISION,
        "duration": DURATION,
        "instrumental": True,
        "inference_steps": 8,
        "guidance_scale": 1.0,
        "shift": 3.0,
        "thinking": False,
    }

    cache_key = hashlib.sha256(
        json.dumps(
            parameters,
            sort_keys=True,
        ).encode()
    ).hexdigest()

    if output.is_file() and manifest.is_file():

        cached = json.loads(
            manifest.read_text()
        )

        if (
            cached.get("cache_key") == cache_key
            and cached.get("output_sha256")
            == hashlib.sha256(
                output.read_bytes()
            ).hexdigest()
        ):

            print(f"[CACHE HIT] {candidate_id}")
            continue

    params = GenerationParams(
        task_type="text2music",
        caption=candidate["caption"],
        lyrics="[Instrumental]",
        instrumental=True,
        bpm=candidate["bpm"],
        duration=DURATION,
        seed=candidate["seed"],
        thinking=False,
        use_cot_metas=False,
        use_cot_caption=False,
        use_cot_language=False,
        inference_steps=8,
        guidance_scale=1.0,
        shift=3.0,
    )

    config = GenerationConfig(
        batch_size=1,
        seeds=[candidate["seed"]],
        use_random_seed=False,
        audio_format="wav",
    )

    record = {
        "timestamp": datetime.now(
            timezone.utc
        ).isoformat(),
        "event": "model_call",
        "stage": "music",
        "provider": "kaggle",
        "model": MODEL,
        "artifact_id": candidate_id,
        "seed": candidate["seed"],
        "parameters": parameters,
        "cost_usd": None,
        "estimated_cost_usd": None,
        "status": "running",
    }

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(0)

    start = time.perf_counter()

    print(
        f"\n[GENERATING] {candidate_id}",
        flush=True,
    )

    try:

        result = generate_music(
            handler,
            None,
            params,
            config,
            save_dir=str(
                OUTPUT_DIR / "raw"
            ),
        )

        if not result.success:

            raise RuntimeError(
                str(result.error)
            )

        audio = result.audios[0]

        shutil.copy2(
            audio["path"],
            output,
        )

        actual_duration = float(
            subprocess.check_output(
                [
                    "ffprobe",
                    "-v", "error",
                    "-show_entries",
                    "format=duration",
                    "-of", "csv=p=0",
                    str(output),
                ],
                text=True,
            ).strip()
        )

        manifest.write_text(
            json.dumps(
                {
                    "cache_key": cache_key,
                    "parameters": parameters,
                    "output_sha256": hashlib.sha256(
                        output.read_bytes()
                    ).hexdigest(),
                    "duration_seconds": actual_duration,
                },
                indent=2,
            )
        )

        record["status"] = "success"
        record["output_path"] = str(output)

        record["duration_seconds"] = actual_duration

    except BaseException as error:

        record["status"] = "failed"
        record["error_type"] = type(error).__name__
        record["error"] = str(error)

        raise

    finally:

        elapsed = time.perf_counter() - start

        record["latency_seconds"] = round(
            elapsed, 3
        )

        record["gpu_minutes"] = round(
            elapsed / 60, 3
        )

        record["peak_vram_gib"] = round(
            torch.cuda.max_memory_allocated(0)
            / 1024**3,
            3,
        )

        with LOG_PATH.open("a") as file:

            file.write(
                json.dumps(record) + "\n"
            )

        print(
            json.dumps(record, indent=2),
            flush=True,
        )

print("\nMusic generation completed.")

In [ ]:
%cd /tmp/ACE-Step-1.5

!MPLBACKEND=Agg CUDA_VISIBLE_DEVICES=0 uv run --no-sync python -u generate_heritage.py

In [ ]:
from pathlib import Path
from IPython.display import Audio, display

MUSIC_DIR = Path(
    "/kaggle/working/heritage-music"
)

for path in sorted(
    MUSIC_DIR.glob("candidate_*.wav")
):
    print(path.name)

    display(
        Audio(filename=str(path))
    )

In [ ]:
import zipfile
from pathlib import Path

music_dir = Path(
    "/kaggle/working/heritage-music"
)

bundle = Path(
    "/kaggle/working/music_outputs.zip"
)

with zipfile.ZipFile(
    bundle,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:

    for path in music_dir.iterdir():

        if path.is_file():

            archive.write(
                path,
                path.name,
            )

print("Bundle:", bundle)

print(
    "Size:",
    round(bundle.stat().st_size / 1024**2, 2),
    "MiB",
)